In [1]:
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
from data import UncertainDataset, ds_single, ds_pressure

In [3]:
from dataclasses import dataclass

@dataclass
class EnsembleSpreadBounds:
    percentile: list[float]
    absolute: list[float]
    relative: list[float]

In [4]:
def compute_ensemble_spread_bounds(mean: xr.DataArray, spread: xr.DataArray, percentile: list[float]) -> EnsembleSpreadBounds:
    absolute = np.nanquantile(spread.where(spread > 0.0), [1-p for p in percentile])

    abs_mean = np.abs(mean)
    rel = spread.where(abs_mean > 0.0) / abs_mean.where(abs_mean > 0.0)
    relative = np.nanquantile(rel.where(rel > 0.0), [1-p for p in percentile])

    return EnsembleSpreadBounds(
        percentile=percentile, absolute=absolute, relative=relative,
    )

In [5]:
from tqdm import tqdm

In [ ]:
with open("ensemble-spread-single.txt", "w") as f:
    for key in tqdm(ds_single.mean):
        print(f"{key}: ", end="", flush=True, file=f)
        print(f"{compute_ensemble_spread_bounds(ds_single.mean[key], ds_single.spread[key], [1.0, 0.99, 0.95])!r}", flush=True, file=f)

 15%|█▌        | 34/226 [06:06<30:13,  9.45s/it]/PUHTI_TYKKY_zwJ6mJ9/miniconda/envs/env1/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1545: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/PUHTI_TYKKY_zwJ6mJ9/miniconda/envs/env1/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1545: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
 27%|██▋       | 60/226 [10:19<29:32, 10.68s/it]/PUHTI_TYKKY_zwJ6mJ9/miniconda/envs/env1/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1545: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


In [ ]:
with open("ensemble-spread-pressure.txt", "w") as f:
    for key in tqdm(ds_pressure.mean):
        print(f"{key}:", flush=True, file=f)
        for pressure in ds_pressure.mean[key].pressure_level.values:
            print(f" - {pressure} hPa: ", end="", flush=True, file=f)
            print(f"{compute_ensemble_spread_bounds(ds_pressure.mean[key].sel(pressure_level=pressure), ds_pressure.spread[key].sel(pressure_level=pressure), [1.0, 0.99, 0.95])!r}", flush=True, file=f)